# Downloading Dataset

In [1]:
!pip install aicrowd-cli

     |████████████████████████████████| 51kB 3.0MB/s 
     |████████████████████████████████| 61kB 4.8MB/s 
     |████████████████████████████████| 81kB 5.3MB/s 
     |████████████████████████████████| 215kB 8.8MB/s 
     |████████████████████████████████| 61kB 4.2MB/s 
     |████████████████████████████████| 163kB 8.9MB/s 
     |████████████████████████████████| 51kB 4.3MB/s 
     |████████████████████████████████| 71kB 5.6MB/s 
ERROR: google-colab 1.0.0 has requirement requests~=2.23.0, but you'll have requests 2.25.1 which is incompatible.
ERROR: datascience 0.10.6 has requirement folium==0.2.1, but you'll have folium 0.8.3 which is incompatible.
  Found existing installation: requests 2.23.0
    Uninstalling requests-2.23.0:
      Successfully uninstalled requests-2.23.0
  Found existing installation: tqdm 4.41.1
    Uninstalling tqdm-4.41.1:
      Successfully uninstalled tqdm-4.41.1


In [2]:
API_KEY = 'cc0a3da7611cfc6098a7bd9db11b3ecf' # Please get your your API Key from [https://www.aicrowd.com/participants/me]
!aicrowd login --api-key $API_KEY

API Key valid
Saved API Key successfully!


In [3]:
# Downloading the Dataset
!mkdir data
!aicrowd dataset download --challenge emotion-detection -j 3 -o data

train.csv: 100% 2.30M/2.30M [00:00<00:00, 4.17MB/s]
test.csv:   0% 0.00/642k [00:00<?, ?B/s]
test.csv: 100% 642k/642k [00:00<00:00, 1.51MB/s]

val.csv: 100% 262k/262k [00:00<00:00, 808kB/s]


# Downloading & Importing Libraries

In [4]:
!pip install --upgrade spacy rich
!python -m spacy download en_core_web_sm # Downloaing the model for engligh language will contains many pretrained preprocessing pipelines

     |████████████████████████████████| 12.8MB 235kB/s 
Requirement already up-to-date: rich in /usr/local/lib/python3.7/dist-packages (10.3.0)
     |████████████████████████████████| 460kB 29.6MB/s 
     |████████████████████████████████| 624kB 36.2MB/s 
     |████████████████████████████████| 51kB 4.9MB/s 
     |████████████████████████████████| 9.1MB 33.5MB/s 
     |████████████████████████████████| 122kB 53.0MB/s 
  Created wheel for smart-open: filename=smart_open-3.0.0-cp37-none-any.whl size=107107 sha256=09f82b1c142cf85fea4b371574288584a3b44bb3ba9e91fe289ee8cf8ae8d1bc
  Stored in directory: /root/.cache/pip/wheels/18/88/7c/f06dabd5e9cabe02d2269167bcacbbf9b47d0c0ff7d6ebcb78
Successfully built smart-open
  Found existing installation: catalogue 1.0.0
    Uninstalling catalogue-1.0.0:
      Successfully uninstalled catalogue-1.0.0
  Found existing installation: srsly 1.0.5
    Uninstalling srsly-1.0.5:
      Successfully uninstalled srsly-1.0.5
  Found existing installation: thinc 

In [5]:
!curl https://raw.githubusercontent.com/tylerneylon/explacy/master/explacy.py -o explacy.py

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  6896  100  6896    0     0  49257      0 --:--:-- --:--:-- --:--:-- 49257


In [6]:
import os
import time
import pandas as pd
import spacy
import explacy
import random
import numpy as np
from xgboost import XGBClassifier
from sklearn.metrics import f1_score, accuracy_score

# To make things more beautiful! 
from rich.console import Console
from rich.table import Table
from rich import pretty
pretty.install()


# Seeding everything for getting same results 
random.seed(1)
spacy.util.fix_random_seed(1)

In [7]:
# spaCy v3.0 the the latest version spaCy 
spacy.__version__

'3.0.6'

# Reading Dataset

In [8]:
train_dataset = pd.read_csv("data/train.csv")
validation_dataset = pd.read_csv("data/val.csv")[1:]
test_dataset = pd.read_csv("data/test.csv")
train_dataset.head(20)

,text,label
0,takes no time to copy/paste a press release,0
1,You're delusional,1
2,Jazz fan here. I completely feel. Lindsay Mann...,0
3,ah i was also confused but i think they mean f...,0
4,Thank you so much. ♥️ that means a lot.,0
5,And I’ll be there!!!,0
6,There are some amazingly cringey compilations ...,0
7,Check the frame (FPS) limit option in the adva...,0
8,you made me think I was in the dbd subreddit w...,0
9,It was in your op.,0


In [9]:
train_dataset['label'].value_counts()

# Text Classification

In [10]:
nlp = spacy.load('en_core_web_sm')

In [11]:
# Getting a sample text from training dataset to demonstrate word2vec  
sample_text = train_dataset.iloc[2]['text'] 
sample_text

"Jazz fan here. I completely feel. Lindsay Mann cousins has more votes than Lindsay Mann, and Lindsay Mann hasn't even stepped on the court this year"

In [12]:
# Inputting the text in nlp function
doc = nlp(sample_text)

# Getting the embeddings from the sample text
doc.vector

array([ 0.5871044 ,  0.10283045,  0.23638554, -0.08171239,  0.02029083,
       -0.12000011, -0.1948305 ,  0.1250714 , -0.01082261, -0.34358275,
       -0.16639529, -0.04950966, -0.01394221,  0.06337869, -0.30135745,
        0.22211872, -0.17156254,  0.03178046,  0.30427024, -0.10826215,
       -0.25342506,  0.30617625, -0.17000276,  0.35598457, -0.00835321,
       -0.11478721, -0.1430562 ,  0.02518663,  0.60922873,  0.19284171,
       -0.23238468, -0.27463096, -0.13183063, -0.27534658,  0.25664884,
        0.05174499,  0.18620381,  0.11441176, -0.10955156,  0.29338667,
       -0.15877348, -0.02914245,  0.1963947 , -0.04410601,  0.12061837,
       -0.0941467 ,  0.27903876, -0.09223508, -0.00497099, -0.25587952,
        0.21098505,  0.01725493, -0.29827487,  0.0894304 ,  0.14340732,
       -0.0376591 , -0.3396481 ,  0.19914041,  0.28556582,  0.18212257,
        0.5140986 ,  0.02056837, -0.18578346, -0.28987882, -0.16651031,
       -0.10539112, -0.05578137,  0.00634063,  0.02737209,  0.14916842,
        0.15076284,  0.31409967, -0.06142968, -0.13555318, -0.08603293,
        0.40901124, -0.07265005, -0.19719984, -0.349496  ,  0.11685906,
        0.20542377,  0.1133521 ,  0.13061962,  0.2739835 , -0.00384022,
       -0.21771309, -0.28375924, -0.41814512, -0.42588463, -0.06813539,
       -0.27145588,  0.17521115, -0.15065633, -0.05529505,  0.06760314,
       -0.1013436 ], dtype=float32)

# Creating our Dataset

In [13]:
def create_data(dataset, is_train=True):

  # If we are using a training dataset
  if is_train == True:

    # Getting all text into a python list
    texts = list(dataset['text'].values)
                 
    # Put the list into the nlp pipeline and converting the output into a list
    preprocessed_texts = list(nlp.pipe(texts))

    # Getting vectors for all texts 
    X = [string.vector  for string in preprocessed_texts]

    # Labels for the corrosponding texts 
    y = dataset['label'].tolist()

    return X, y

  else:

    # Getting all text into a python list
    texts = list(dataset['text'].values)
                 
    # Put the list into the nlp pipeline and converting the output into a list
    preprocessed_texts = list(nlp.pipe(texts))

    # Getting vectors for all texts 
    X = [string.vector  for string in preprocessed_texts]

    return X

In [14]:
# Creating the training dataset
start_time = time.time()
X_train, y_train = create_data(train_dataset)
print("Elapsed time: %s seconds" % round(time.time() - start_time, 4))

# Creating the validation dataset
start_time = time.time()
X_val, y_val = create_data(validation_dataset)
print("Elapsed time: %s seconds" % round(time.time() - start_time, 4))

X_train[0], y_train[0]

Elapsed time: 66.8408 seconds
Elapsed time: 6.9152 seconds


(
    array([ 0.14151856, -0.05933758, -0.08044346, -0.1107378 ,  0.11325426,
        0.15893325, -0.44572473,  0.21314225,  0.07863858, -0.12423076,
        0.08870672, -0.16083065,  0.03217425, -0.18913928, -0.43932933,
        0.614143  , -0.04348904,  0.15507331, -0.10762676, -0.6841317 ,
       -0.52705824,  0.19307005,  0.5150874 , -0.78364027, -0.5798768 ,
       -0.36855024,  0.261396  ,  0.08007441,  0.13464396, -0.11683945,
       -0.40335947, -0.4810744 , -0.1996509 , -0.40538788,  0.907061  ,
       -0.08425845,  0.41184407, -0.05256712, -0.01928839,  0.67991185,
        0.18288395, -0.00932413,  0.15208176,  0.5218999 , -0.21960959,
       -0.0870916 ,  0.0571377 ,  0.39526838,  0.11505216,  0.03575191,
        0.18940884,  0.35989684, -0.03341857,  0.50211823,  0.25563306,
       -0.45388022,  0.04130013,  0.1111506 ,  0.11391255,  0.12924205,
       -0.44648582, -0.04701398, -0.12538372,  0.11663225,  0.3620197 ,
       -0.00661604, -0.25024778, -0.3998903 , -0.07583453,  0.747198  ,
        0.5959583 , -0.17592818, -0.04306039, -0.52941674, -0.12472894,
       -0.43728942, -0.06499843,  0.0685156 ,  0.23714057, -0.20262413,
        0.41856593,  0.01466806, -0.19088936,  0.36778176,  0.09763627,
       -0.3632294 , -0.15831968, -0.43174067, -0.08282916, -0.09869532,
        0.14616643,  0.02205803, -0.33136448,  0.19892709, -0.50562334,
       -0.14163868], dtype=float32),
    0
)

# Training the Model

In [23]:
X_train = np.array(X_train)
X_val = np.array(X_val)
y_train = np.array(y_train)
y_val = np.array(y_val)

print(X_train.shape)
print(X_val.shape)
print(y_train.shape)
print(y_val.shape)

(31255, 96)
(3472, 96)
(31255,)
(3472,)


In [24]:
model = XGBClassifier(
    n_estimators=1000, subsample=0.8, colsample_bytree=0.8,
    objective='binary:logistic', scale_pos_weight=4,
    seed=27
)
eval_set = [(X_val, y_val)]

model.fit(X_train, y_train, early_stopping_rounds=50,
          eval_metric="auc", eval_set=eval_set, verbose=True)

[0]	validation_0-auc:0.559508
Will train until validation_0-auc hasn't improved in 50 rounds.
[1]	validation_0-auc:0.569899
[2]	validation_0-auc:0.569146
[3]	validation_0-auc:0.572955
[4]	validation_0-auc:0.572721
[5]	validation_0-auc:0.576952
[6]	validation_0-auc:0.576486
[7]	validation_0-auc:0.577246
[8]	validation_0-auc:0.580862
[9]	validation_0-auc:0.582619
[10]	validation_0-auc:0.582515
[11]	validation_0-auc:0.585837
[12]	validation_0-auc:0.588232
[13]	validation_0-auc:0.587736
[14]	validation_0-auc:0.590206
[15]	validation_0-auc:0.591812
[16]	validation_0-auc:0.591587
[17]	validation_0-auc:0.593207
[18]	validation_0-auc:0.594562
[19]	validation_0-auc:0.595734
[20]	validation_0-auc:0.597291
[21]	validation_0-auc:0.597657
[22]	validation_0-auc:0.599209
[23]	validation_0-auc:0.600187
[24]	validation_0-auc:0.602087
[25]	validation_0-auc:0.600884
[26]	validation_0-auc:0.60205
[27]	validation_0-auc:0.601949
[28]	validation_0-auc:0.602843
[29]	validation_0-auc:0.606184
[30]	validation_0

XGBClassifier(base_score=0.5, booster='gbtree', colsample_bylevel=1,
              colsample_bynode=1, colsample_bytree=0.8, gamma=0,
              learning_rate=0.1, max_delta_step=0, max_depth=3,
              min_child_weight=1, missing=None, n_estimators=1000, n_jobs=1,
              nthread=None, objective='binary:logistic', random_state=0,
              reg_alpha=0, reg_lambda=1, scale_pos_weight=4, seed=27,
              silent=None, subsample=0.8, verbosity=1)

# Evaluation

In [26]:
y_pred = model.predict(X_train)

# Getting F1 & Accuracy score of training predictions
f1 = f1_score(y_train, y_pred)
accuracy = accuracy_score(y_train, y_pred)

print(f"Training F1 Score  : {round(f1, 4)} and Accuracy Score {round(accuracy, 4)}")

Training F1 Score  : 0.5325 and Accuracy Score 0.7017


In [28]:
y_pred = model.predict(X_val)

# Getting F1 & Accuracy score of validation predictions
f1 = f1_score(y_val, y_pred)
accuracy = accuracy_score(y_val, y_pred)

print(f"Validation F1 Score  : {round(f1, 4)} and Accuracy Score {round(accuracy, 4)}")

Validation F1 Score  : 0.3738 and Accuracy Score 0.6005


# Submitting Results

In [29]:
# By settings is_train=False, the create_data function will only output the features as setuped in the function

start_time = time.time()
test_data = create_data(test_dataset, is_train=False)
test_data = np.array(test_data)

test_predictions = model.predict(test_data)
print("Elapsed time: %s seconds" % round(time.time() - start_time, 4))

Elapsed time: 17.8621 seconds


In [30]:
# Applying the predictions to the labels column of the sample submission 
test_dataset['label'] = test_predictions
print(test_dataset.shape)

test_dataset.head(20)

(8682, 2)


,text,label
0,I was already over the edge with Cassie Zamora...,0
1,I think you're right. She has oodles of cash a...,1
2,Haha I love this. I used to give mine phone bo...,0
3,Probably out of desperation as they going no a...,1
4,Sorry !! You’re real good at that!!,0
5,I say we get the pitch forks and make him have...,0
6,He looks really different now.,0
7,I swear people just want to be angry,1
8,lol robot car,0
9,Yeah I’ve been watching these videos and they ...,1


In [31]:
!mkdir assets

# Saving the sample submission in assets directory
test_dataset.to_csv(os.path.join("assets", "submission.csv"), index=False)

# Uploading the Results

In [32]:
!aicrowd notebook submit -c emotion-detection -a assets --no-verify

Mounting Google Drive 💾
Your Google Drive will be mounted to access the colab notebook
Go to this URL in a browser: https://accounts.google.com/o/oauth2/auth?client_id=947318989803-6bn6qk8qdgf4n4g3pfee6491hc0brc4i.apps.googleusercontent.com&redirect_uri=urn%3aietf%3awg%3aoauth%3a2.0%3aoob&scope=email%20https%3a%2f%2fwww.googleapis.com%2fauth%2fdocs.test%20https%3a%2f%2fwww.googleapis.com%2fauth%2fdrive%20https%3a%2f%2fwww.googleapis.com%2fauth%2fdrive.photos.readonly%20https%3a%2f%2fwww.googleapis.com%2fauth%2fpeopleapi.readonly%20https%3a%2f%2fwww.googleapis.com%2fauth%2fdrive.activity.readonly%20https%3a%2f%2fwww.googleapis.com%2fauth%2fexperimentsandconfigs%20https%3a%2f%2fwww.googleapis.com%2fauth%2fphotos.native&response_type=code

Enter your authorization code:
4/1AY0e-g5pAPPVY6MNUc2r_h0nM50g-DIq6ktoR1RHlwNZ7rGWbiE57hHYZM8
Mounted at /content/drive
Using notebook: /content/drive/MyDrive/Colab Notebooks/embedding-xgboost-classifier.ipynb for submission...
Scrubbing API keys from t